In [ ]:
import pandas as pd
import math
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from anytree import Node, RenderTree
from tabulate import tabulate 


MEMBACA FILE EXCEL

In [ ]:
#Membaca file-Excel
df = pd.read_excel('data_donor_lagi.xlsx')
df 

In [ ]:
df.info() 

PREPROCESSING

In [ ]:
#Membaca file-Excel
df = pd.read_excel('data_donor_lagi.xlsx')

#Melakukan Proses Bining
#Bining Umur
umur_bins = [0,25,45,np.inf]
umur_labels = ['Umur<=25','25<Umur<=45','45<Umur']
df['Umur'] = pd.cut(df['Umur'], bins= umur_bins, labels= umur_labels, right= True)

#Bining HB
hb_pria = [
    (df['Jenis Kelamin'] == 'Pria') & (df['HB'].between(13.0, 17.0)),
    (df['Jenis Kelamin'] == 'Pria') & ~df['HB'].between(13.0, 17.0)
]
labels_hb_pria = ['Normal', 'Tidak Normal']

hb_wanita = [
    (df['Jenis Kelamin'] == 'Wanita') & (df['HB'].between(12.0, 15.0)),
    (df['Jenis Kelamin'] == 'Wanita') & ~df['HB'].between(12.0, 15.0)
]
labels_hb_wanita = ['Normal', 'Tidak Normal']

hb_bins = hb_pria + hb_wanita
labels =  labels_hb_pria + labels_hb_wanita
df['HB'] = np.select(hb_bins, labels, default='Tidak Diketahui')

#Bining Tensi
def bins_tensi (s,d):
    if s < 120 or d < 80 :
        return 'Rendah'
    elif s > 139 or d > 89 :
        return 'Tinggi'
    else :
        return 'Normal'

pos = df.columns.get_loc('Tensi S')    
df.insert(pos, 'Tensi', df.apply(lambda row : bins_tensi(row['Tensi S'], row['Tensi D']), axis= 1))
df.drop(columns=['Tensi S','Tensi D'],inplace= True)

#Bining_Berat Badan
bb_bins = [0,60,80,np.inf]
bb_labels = ['BB<=60','60<BB<=80','80<BB']
df['Berat Badan'] = pd.cut(df['Berat Badan'], bins= bb_bins, labels= bb_labels, right= True)
df

PENGAMBILAN DATA TRAIN DAN DATA TEST

In [ ]:
x = df.drop(columns=['Status'])
y = df['Status']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2,random_state=42, stratify=y)

train_df = x_train.copy()
train_df['Label'] = y_train
test_df = x_test.copy()
test_df['Label'] = y_test

train_data = train_df.to_dict(orient='records')
test_data = test_df.to_dict(orient='records') 

In [ ]:
full_attr_values = {
    attr: sorted(df[attr].dropna().unique().tolist())
    for attr in df.columns if attr != 'Label'
}

MENGHITUNG ENTROPY

In [ ]:
def entropy(data, target_attr):
    total_entropy = len(data)
    if total_entropy == 0:
        return 0

    counts = Counter(record[target_attr] for record in data)
    ent = 0
    for count in counts.values():
        p = count / total_entropy
        ent -= p * math.log2(p)
    return ent 

MENGHITUNG GAIN

In [ ]:
def info_gain(data, attr, target_attr):
    total = len(data)
    subsets = {}
    for record in data:
        key = record[attr]
        subsets.setdefault(key, []).append(record)
    subset_entropy = 0
    for subset in subsets.values():
        p = len(subset) / total
        subset_entropy += p * entropy(subset, target_attr)
    return entropy(data, target_attr) - subset_entropy 

MENGHTUNG SPLIT INFO

In [ ]:
def split_info(data, attr):
    total = len(data)
    counts = Counter(record[attr] for record in data)
    si = 0
    for count in counts.values():
        p = count / total
        si -= p * math.log2(p) if p > 0 else 0
    return si 

MENGHITUNG GAIN RATIO

In [ ]:
def gain_ratio(data, attr, target_attr):
    ig = info_gain(data, attr, target_attr)
    si = split_info(data, attr)
    if si == 0:
        return 0
    return ig / si 

MEMILIH GAIN RATIO TERBAIK

In [ ]:
def choose_best_attribute(data, attributes, target_attr):
    global iterasi_ke
    try:
        iterasi_ke += 1
    except:
        iterasi_ke = 1

    print(f'\n==== Iterasi {iterasi_ke} ====')

    ent_dataset = entropy(data, target_attr)
    print(f'Entropy Dataset: {ent_dataset:.4f}\n')

    best_attr = None
    best_gr = -1
    total = len(data)

    for attr in attributes:
        ent_attr = 0
        value_details = []
        values = set(record[attr] for record in data)
        for val in values:
            subset = [record for record in data if record[attr] == val]
            weight = len(subset) / total
            ent_val = entropy(subset, target_attr)
            ent_attr += weight * ent_val
            value_details.append([val, len(subset), round(ent_val, 4)])

        si = split_info(data, attr)
        gr = (ent_dataset - ent_attr) / si if si != 0 else 0

        print(f'[{attr}]')
        print(tabulate(value_details, headers=['Nilai', 'Jumlah', 'Entropy'], tablefmt='plain'))
        print(f'Entropy Atribut: {ent_attr:.4f}')
        print(f'Gain Ratio     : {gr:.4f}\n')

        if gr > best_gr:
            best_gr = gr
            best_attr = attr

    print(f'=> Atribut terbaik yang dipilih di iterasi {iterasi_ke}: {best_attr}')
    return best_attr 

PROSES TREE

In [ ]:
def majority_class(data, target_attr):
    counts = Counter(record[target_attr] for record in data)
    return counts.most_common(1)[0][0]

def build_tree(data, attributes, target_attr):
    labels = [record[target_attr] for record in data]
    if len(set(labels)) == 1:
        return labels[0]
    if not attributes:
        return majority_class(data, target_attr)

    best_attr = choose_best_attribute(data, attributes, target_attr)
    tree = {best_attr: {}}

    attr_values = full_attr_values.get(best_attr, set(record[best_attr] for record in train_data))
    for val in attr_values:
        subset = [record for record in data if record[best_attr] == val]
        if not subset:
            tree[best_attr][val] = majority_class(data, target_attr)
        else:
            new_attrs = [a for a in attributes if a != best_attr]
            subtree = build_tree(subset, new_attrs, target_attr)
            tree[best_attr][val] = subtree

    child_vals = list(tree[best_attr].values())
    if all (not isinstance(v, dict) and v == child_vals[0] for v in child_vals):
        return child_vals[0]
    
    return tree 

In [ ]:
def predict(tree, record):
    while isinstance (tree, dict):
        attr = next(iter(tree))
        value = record.get(attr)
        if value not in tree[attr] :
            return majority_class([record],'Label')
        tree = tree [attr][value]
    return tree 

In [ ]:
def build_anytree(tree, parent=None):
    if not isinstance(tree, dict):
        return Node(str(tree), parent=parent)
    attr = next(iter(tree))
    node = Node(str(attr), parent=parent)
    for val, subtree in tree[attr].items():
        child = build_anytree(subtree, parent=node)
        child.name = f'{val} : {child.name}'
    return node

VISUALISASI TREE

In [ ]:
iterasi_ke = 0
attributes = list(train_df.columns)
attributes.remove('Label') 
decision_tree = build_tree(train_data, attributes, 'Label')

root = build_anytree(decision_tree)
for pre, fill, node in RenderTree(root):
    print(f'{pre}{node.name}') 

CEK AKURASI

In [ ]:
correct = 0
for record in train_data:
    pred = predict(decision_tree, record)
    actual = record['Label']
    if pred == actual:
        correct += 1

accuracy = correct / len(train_data)
print(f'\nAkurasi Data Train: {accuracy*100:.2f}%') 

In [ ]:
correct = 0
for record in test_data:
    pred = predict(decision_tree, record)
    actual = record['Label']
    if pred == actual:
        correct += 1

accuracy = correct / len(test_data)
print(f'Akurasi Data Test: {accuracy*100:.2f}%') 